# MS-MARCO: BGE-Reranker-v2-m3 Complete Scoring

**Generates and publishes TWO datasets:**
1. `msmarco-bge-reranker-v2-m3-7hn-raw` - Raw logits (published FIRST)
2. `msmarco-bge-reranker-v2-m3-7hn-softmax` - Softmax probabilities τ=1.0 (published SECOND)

**Model**: `BAAI/bge-reranker-v2-m3`
**Source**: `satyam2025/msmarco-7hn-rolling-plain`

In [ ]:
# CELL 1: Environment
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import torch
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

print(f"✅ CUDA: {torch.cuda.is_available()}, GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'N/A'}")

In [ ]:
# CELL 2: Install
!pip install -q FlagEmbedding transformers>=4.48.0 datasets huggingface-hub tqdm accelerate
print("✅ Dependencies installed")

In [ ]:
# CELL 3: Imports
import json
from pathlib import Path
from dataclasses import dataclass
from typing import List, Dict

import torch
import torch.nn.functional as F
from tqdm.auto import tqdm
from datasets import Dataset, load_dataset
from huggingface_hub import login as hf_login
from kaggle_secrets import UserSecretsClient
from FlagEmbedding import FlagReranker

print(f"PyTorch: {torch.__version__}")

In [ ]:
# CELL 4: Configuration
@dataclass
class Config:
    teacher_model: str = "BAAI/bge-reranker-v2-m3"
    max_length: int = 512
    batch_size: int = 128
    num_hard_negatives: int = 7
    checkpoint_every: int = 50000
    softmax_temperature: float = 1.0  # STANDARD SOFTMAX - real probability distribution
    hf_username: str = "satyam2025"
    source_dataset: str = "satyam2025/msmarco-7hn-rolling-plain"
    raw_dataset: str = "msmarco-bge-reranker-v2-m3-7hn-raw"
    softmax_dataset: str = "msmarco-bge-reranker-v2-m3-7hn-softmax"

config = Config()

try:
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
    print("✅ HF token loaded")
except:
    HF_TOKEN = "HF_Token-goes-here"

print(f"Model: {config.teacher_model}")
print(f"Softmax τ: {config.softmax_temperature} (standard softmax)")
print(f"Output: {config.hf_username}/{config.raw_dataset}")
print(f"Output: {config.hf_username}/{config.softmax_dataset}")

In [ ]:
# CELL 5: Paths
BASE_PATH = Path("/kaggle/working")
PATHS = {'checkpoints': BASE_PATH / 'checkpoints', 'datasets': BASE_PATH / 'datasets'}
for p in PATHS.values(): p.mkdir(parents=True, exist_ok=True)
CHECKPOINT_PATH = PATHS['checkpoints'] / "msmarco_bge_checkpoint.json"
print(f"📁 Base: {BASE_PATH}")

In [ ]:
# CELL 6: Load Dataset
print(f"📥 Loading: {config.source_dataset}")
hf_login(token=HF_TOKEN)
ds = load_dataset(config.source_dataset, split='train')
print(f"✅ Loaded {len(ds):,} rows")

In [ ]:
# CELL 7: Load Model
print(f"📥 Loading: {config.teacher_model}")
reranker = FlagReranker(config.teacher_model, use_fp16=True, device='cuda')
print("✅ Model loaded")

In [ ]:
# CELL 8: Test
test_pairs = [['What is the capital of France?', 'Paris is the capital.'], ['What is the capital of France?', 'London is in England.']]
test_scores = reranker.compute_score(test_pairs)
test_softmax = F.softmax(torch.tensor(test_scores, dtype=torch.float64), dim=0).tolist()
print(f"🧪 Raw: {test_scores[0]:.4f} vs {test_scores[1]:.4f}")
print(f"🧪 Softmax (τ=1): {test_softmax[0]:.6f} vs {test_softmax[1]:.6f} (sum={sum(test_softmax):.4f})")

In [ ]:
# CELL 9: Generate RAW Scores
def generate_raw_scores(dataset, config):
    raw_data, start_idx = [], 0
    if CHECKPOINT_PATH.exists():
        with open(CHECKPOINT_PATH) as f: cp = json.load(f)
        raw_data, start_idx = cp['raw_data'], cp['last_idx'] + 1
        print(f"📂 Resuming from {start_idx:,}")
    
    if start_idx >= len(dataset): return raw_data
    
    print(f"🚀 Scoring {len(dataset)-start_idx:,} rows...")
    for batch_start in tqdm(range(start_idx, len(dataset), config.batch_size), desc="Scoring"):
        batch_end = min(batch_start + config.batch_size, len(dataset))
        batch = dataset[batch_start:batch_end]
        
        all_pairs = []
        for i in range(len(batch['query'])):
            all_pairs.append([batch['query'][i], batch['positive'][i]])
            for j in range(1, config.num_hard_negatives + 1):
                all_pairs.append([batch['query'][i], batch[f'neg_{j}'][i]])
        
        all_scores = reranker.compute_score(all_pairs)
        if isinstance(all_scores, (int, float)): all_scores = [all_scores]
        
        scores_per_row = 1 + config.num_hard_negatives
        for i in range(len(batch['query'])):
            start = i * scores_per_row
            row_scores = all_scores[start:start + scores_per_row]
            raw_data.append({
                'query': batch['query'][i],
                'positives': {'passage': [batch['positive'][i]], 'score': [float(row_scores[0])]},
                'negatives': {'passage': [batch[f'neg_{j}'][i] for j in range(1, config.num_hard_negatives + 1)], 'score': [float(s) for s in row_scores[1:]]}
            })
        
        if len(raw_data) % config.checkpoint_every < config.batch_size:
            with open(CHECKPOINT_PATH, 'w') as f: json.dump({'last_idx': batch_end - 1, 'raw_data': raw_data}, f)
    
    return raw_data

msmarco_raw_data = generate_raw_scores(ds, config)
print(f"✅ Generated {len(msmarco_raw_data):,} raw scores")

In [ ]:
# CELL 10: Publish RAW Logits
print("📤 Publishing RAW LOGITS...")
raw_hf = Dataset.from_list(msmarco_raw_data)
raw_name = f"{config.hf_username}/{config.raw_dataset}"
raw_hf.push_to_hub(raw_name, token=HF_TOKEN, private=False)
print(f"✅ RAW PUBLISHED: https://huggingface.co/datasets/{raw_name}")

In [ ]:
# CELL 11: Convert to Softmax (τ=1.0)
print(f"🔄 Converting to softmax (τ={config.softmax_temperature})...")
msmarco_softmax_data = []
for item in tqdm(msmarco_raw_data, desc="Converting"):
    all_scores = [item['positives']['score'][0]] + list(item['negatives']['score'])
    # STANDARD SOFTMAX (τ=1.0) with float64 for precision
    probs = F.softmax(torch.tensor(all_scores, dtype=torch.float64), dim=0).tolist()
    msmarco_softmax_data.append({
        'query': item['query'],
        'positives': {'passage': item['positives']['passage'], 'score': [probs[0]]},
        'negatives': {'passage': item['negatives']['passage'], 'score': probs[1:]}
    })

print(f"\n📌 Sample (should show REAL probabilities, not 1s and 0s):")
for i in [0, 1, 2]:
    s = msmarco_softmax_data[i]
    print(f"   Row {i}: pos={s['positives']['score'][0]:.6f}, negs={[f'{p:.6f}' for p in s['negatives']['score'][:3]]}...")

In [ ]:
# CELL 12: Publish SOFTMAX
print("📤 Publishing SOFTMAX PROBABILITIES...")
softmax_hf = Dataset.from_list(msmarco_softmax_data)
softmax_name = f"{config.hf_username}/{config.softmax_dataset}"
softmax_hf.push_to_hub(softmax_name, token=HF_TOKEN, private=False)
print(f"✅ SOFTMAX PUBLISHED: https://huggingface.co/datasets/{softmax_name}")

In [ ]:
# CELL 13: Summary
print("\n" + "="*60)
print("🎉 COMPLETE!")
print("="*60)
print(f"Model: {config.teacher_model}")
print(f"Rows: {len(msmarco_raw_data):,}")
print(f"\n📦 RAW: https://huggingface.co/datasets/{config.hf_username}/{config.raw_dataset}")
print(f"📦 SOFTMAX: https://huggingface.co/datasets/{config.hf_username}/{config.softmax_dataset}")
print(f"\nSoftmax τ={config.softmax_temperature} → Real probability distribution")